# Session 13 · Data integration I

Module 3.1, Data Integration. You are newly arrived engineers in the road safety unit of the
*Demarcación de Carreteras del Estado en Castilla y León Oriental*, which covers the state roads
of Burgos and Soria among others. The unit has to answer one question:

> **Where and when is accident risk highest on our roads?**

No single file answers it. Today we look at the files one at a time, then at how they fit
together.

Work individually or in pairs. Each task has a code (`1.1`, `1.2`…) that the slides use too.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

# Runs as is locally (data/ next to the notebook) and on Colab (data read from GitHub).
DATA = "data/" if os.path.isdir("data") else "https://raw.githubusercontent.com/daniel-verdugo/upm-data-integration/main/3.1-data-integration/session13/data/"

---
## Block 1 · The table

The planning section prepares the 2019 traffic table for its permanent counting stations.

| File | Content |
|---|---|
| `traffic/stations.csv` | Station catalogue: station, road, PK, and the IMD 2018 as published. |
| `traffic/hourly_counts.csv` | Counter output: vehicles per station, direction and hour. Hours when the counter failed are missing. |
| `traffic/hourly_corrections.csv` | The hours the counters missed, re-estimated by the traffic office, with the cause of the fault. |

Counters plus corrections make the complete year.

In [ ]:
stations = pd.read_csv(DATA + "traffic/stations.csv")
counts = pd.read_csv(DATA + "traffic/hourly_counts.csv", parse_dates=["date"])
corrections = pd.read_csv(DATA + "traffic/hourly_corrections.csv", parse_dates=["date"])

In [ ]:
# Given: a first look at each table
display(stations.head(3), counts.head(3), corrections.head(3))
print(len(stations), "stations ·", len(counts), "counter rows ·", len(corrections), "corrected rows")

### Task 1.1 · Know the tables

- **a.** For each table, write in one sentence what one row is.
- **b.** Prove its key in code: replace `FILL_IN` in `key` and run the cell. Zero duplicates means the key holds.
- **c.** Predict: on a mature network with no major changes, how much does a station's IMD change
   from one year to the next? Write down a range.

In [ ]:
# a. One row of stations is ...   One row of counts is ...   One row of corrections is ...

# b. Fill in the columns that identify one row of counts and corrections
key = [FILL_IN]
print("stations, station unique:", stations.station.is_unique)
print("counts, duplicated rows:", counts.duplicated(key).sum())
print("corrections, duplicated rows:  ", corrections.duplicated(key).sum())

# c. My prediction: from ___ % to ___ %

<details><summary><b>Hint 1</b></summary>

`df.duplicated(cols).sum()` counts the rows whose values in `cols` already appeared in an earlier row. Try `key = ["station", "date"]` first and see how many duplicates you get.

</details>

<details><summary><b>Hint 2</b></summary>

Both tables write one row per station, day, hour and **direction** — `counts` from the counter itself, `corrections` for the hours it missed.

</details>

### Task 1.2 · Build the traffic table

- **a.** *Given:* the cell below stacks counters and corrections into one hourly table, `hourly`,
   with `veh = light_vehicles + heavy_vehicles`, and checks it. It also defines `station_table(daily)`,
   which turns a daily table into the station table (IMD, share of heavy vehicles, change
   from 2018). Run it.
- **b.** **Your part.** The office dashboard reads a daily table, `daily`, with columns
   `station, road, date, light_vehicles, heavy_vehicles, veh`. Build it.
- **c.** Run `station_table(daily)`.

The traffic office's analyst sends you the paragraph for the report:

> *"Traffic was stable across the network in 2019 (up to about +3%), except at two
> stations on the A-1, BU-02 and BU-04, which recorded drops of 17–18%. Possible causes
> include traffic shifting to alternative routes."*

- **d.** Do your numbers agree? **Would you sign it?** Yes or no, and one sentence why.

In [ ]:
# Given: the hourly table and its checks
key = ["station", "date", "hour", "direction"]
hourly = pd.concat([counts, corrections.drop(columns="reason")], ignore_index=True)
hourly["veh"] = hourly.light_vehicles + hourly.heavy_vehicles
print("duplicated hours:", hourly.duplicated(key).sum())
print("hours per station:", hourly.groupby("station").size().unique(), "(365 days × 24 h × 2 directions = 17520)")


# Given: from a daily table to the station table
def station_table(daily):
    t = daily.groupby("station").agg(imd=("veh", "mean"), heavy_vehicles=("heavy_vehicles", "sum"), veh=("veh", "sum"))
    t = stations.merge(t.reset_index(), on="station", validate="1:1")
    t["heavy_vehicles_%"] = (t.heavy_vehicles / t.veh * 100).round(1)
    t["var_2018_%"] = ((t.imd / t.imd_2018 - 1) * 100).round(1)
    t["imd"] = t.imd.round().astype(int)
    return t[["station", "road", "pk", "imd", "imd_2018", "heavy_vehicles_%", "var_2018_%"]]

In [ ]:
# b. Build daily, with columns station, road, date, light_vehicles, heavy_vehicles, veh
daily = FILL_IN
daily.head()

# c.
station_table(daily)

# d. Sign the paragraph? ...

<details><summary><b>Hint 1</b></summary>

A daily table adds up the hours of each day: `hourly.groupby(FILL_IN, as_index=False)[["light_vehicles", "heavy_vehicles", "veh"]].sum()`. `groupby(cols, as_index=False)` makes one row per combination of values in `cols` and keeps those columns in the result.

</details>

<details><summary><b>Hint 2</b></summary>

Compare the `var_2018_%` column with your prediction from task 1.1.

</details>

#### New information: the analyst's pipeline

The traffic office sends the code behind the paragraph. It is Idea A.


In [ ]:
# Given: the analyst's own pipeline (Idea A), the office's pipeline behind the paragraph
daily_office = hourly.groupby(["station", "road", "date"], as_index=False)[["light_vehicles", "heavy_vehicles", "veh"]].sum()

### Task 1.3 · Count what was averaged

IMD is a mean over days. Work on `daily_office`, the analyst's daily table.

- **a.** How many days were averaged for each station? How many should there be?
- **b.** Each row of the daily table should be one station and day (its grain). Count the rows that
   break it, and look at one station-day that does.
- **c.** Is `road` constant for each station?

In [ ]:
# a. rows (days) per station
daily_office.groupby("station").FILL_IN()

In [ ]:
# b. rows that repeat a station and day
dup = daily_office[daily_office.duplicated([FILL_IN], keep=False)]
print(len(dup), "rows share a station and day")
dup.sort_values(["station", "date"]).head(4)

In [ ]:
# c. number of different road codes per station, in hourly
hourly.groupby("station").road.FILL_IN()

<details><summary><b>Hint 1</b></summary>

a: `.size()` counts rows per group. c: `.nunique()` counts distinct values per group.

</details>

<details><summary><b>Hint 2</b></summary>

b: `keep=False` marks **every** copy of a repeated key, not only the second one, so you can see both rows side by side.

</details>

---
## Block 2 · The columns

The road safety unit asks about eight corridors in Soria and Burgos, 2016–2019:

> **Is the accident risk higher at night? Give the accident rate per 10⁸ veh·km by hour of
> day and by day of the week.**

| File | Content |
|---|---|
| `hourly/segments.csv` | One row per corridor: road, PK range, its counting station, mean IMD 2016–2019. |
| `hourly/hourly_profile.csv` | Hourly profile per counting station, extracted for this study by the traffic office from its counter database. |
| `hourly/accidents.csv` | Accidents with victims on the eight corridors, from the DGT microdata. |

**Dictionary of `hourly_profile.csv`**, as supplied by the traffic office:

| Campo | Descripción |
|---|---|
| `station` | Código de la estación de aforo |
| `year` | Año |
| `weekday` | Día de la semana |
| `hour` | Hora |
| `volume` | Vehículos por hora |

**Dictionary of `accidents.csv`**, extract of the DGT dictionary:

| Campo | Descripción |
|---|---|
| `ACCIDENT_ID` | Identificador del accidente |
| `YEAR`, `MONTH` | Año y mes del accidente |
| `WEEKDAY` | 1 LUNES, 2 MARTES, 3 MIÉRCOLES, 4 JUEVES, 5 VIERNES, 6 SÁBADO, 7 DOMINGO |
| `HOUR` | Hora del accidente (0–23) |
| `ROAD`, `KM` | Carretera y punto kilométrico |
| `TOTAL_DEATHS_30D` | Fallecidos a 30 días |
| `TOTAL_VICTIMS_30D` | Víctimas a 30 días |

In [ ]:
segments = pd.read_csv(DATA + "hourly/segments.csv")
profile = pd.read_csv(DATA + "hourly/hourly_profile.csv")
accidents = pd.read_csv(DATA + "hourly/accidents.csv")

### Task 2.1 · Read the dictionaries


**No code.**

- **a.** For each column of `hourly_profile.csv`, write one line saying what you understand it
   to contain.
- **b.** Which columns will you use to join the profile with the accidents?
- **c.** Predict: the order of magnitude of the accident rate (accidents with victims per 10⁸
   veh·km) on interurban state roads, and how it varies through the night.

In [ ]:
# your code here

### Task 2.2 · Compute the rates

The joined table `p` with profile and corridor lengths is given:

In [ ]:
segments["long_km"] = segments.end_km - segments.start_km
p = profile.merge(segments, on="station", validate="m:1")
p.head()

**Helpers** · 

Given: the accident counts, and the `attach_accidents`/`rate` functions you'll use in **b**, and in every idea below.

In [ ]:
# accident counts per weekday/hour, using the DGT's own codes (WEEKDAY, HOUR)
n = accidents.groupby(["WEEKDAY", "HOUR"]).size().rename("accidents").reset_index()

def compute_rate(d):
    # accidents per 1e8 veh·km, summed over the whole table -- a single number
    return d.accidents.sum() / d.vehkm.sum() * 1e8

def rate_by(d, key):
    # same rate, but one value per group (e.g. key="hour" or key="weekday")
    g = d.groupby(key)[["accidents", "vehkm"]].sum()
    return g.accidents / g.vehkm * 1e8

def attach_accidents(expo, how="left"):
    # left-join: keep every row of expo, attach accidents where the codes match
    # naive match on expo's own weekday/hour -- do they mean the same thing as WEEKDAY/HOUR?
    d = expo.merge(n, left_on=["weekday", "hour"], right_on=["WEEKDAY", "HOUR"], how=how)
    # rows with no matching accidents get 0, not NaN
    return d.assign(accidents=d.accidents.fillna(0))

Your task is to:
- **a.** Build the exposure in veh·km for each weekday and hour, over the eight corridors and
   the period of the accidents.
- **b.** Count accidents per weekday and hour, join them to the exposure, and compute the
   overall rate, the rate by hour (plot it) and the rate by weekday.
- **c.** Does your overall rate match your prediction?
- **d.** The head of the traffic patrols asks: **at which night hours should the patrols
   concentrate?** Write a one-line recommendation based on your results.

In [ ]:
# a. exposure in veh·km, for each weekday and hour, over the eight corridors and the accidents' period
expo = FILL_IN
expo.head()

<details><summary><b>Hint</b></summary>

`p["vehkm"] = FILL_IN` — one row's share of veh·km, from `p.volume` and `p.long_km`.
`expo = p.groupby(["weekday", "hour"], as_index=False).vehkm.sum()` sums it per weekday and hour.

</details>

In [ ]:
# b. join to the accident counts and compute the rates

# exposure + matching accident counts, one row per weekday/hour
df = attach_accidents(expo)

# single number: accidents per 1e8 veh·km, across everything
overall_rate = compute_rate(df)
print(f"overall: {overall_rate:.2f} per 1e8 veh·km")

# one rate per hour of day
rate_by_hour = rate_by(df, "hour")
rate_by_hour.plot(marker="o", title="Accident rate by hour")

# one rate per day of week
rate_by_weekday = rate_by(df, "weekday")
rate_by_weekday

In [ ]:
# c. Does your overall rate match your prediction from 2.1? ...

# d. Recommendation for the head of the patrols: ...

#### New information: email from the traffic office

> *"Las horas van de 1 a 24: la hora 8 es de 7:00 a 8:00. Los equipos registran el final de
> cada intervalo."*

(Hours run from 1 to 24: hour 8 is 7:00 to 8:00. The equipment records the end of each
interval.)

### Task 2.3 · Test the dictionary


`hour`'s label is an assumption about the data. Test it.

- Compare `hour`'s range with `HOUR`. Join the two on hour alone with `indicator=True`.
  What is left unmatched on each side?

In [ ]:
# your code here

### Task 2.4 · Night and day


With the corrected exposure, a colleague drafts one more line for the report:

> *"At night, from 22:00 to 06:00, the accident rate on our corridors is about four times the daytime rate (08:00–20:00)."*

Check it. **Would you sign it?**

In [ ]:
# your code here

---
## Block 3 · The relations

Next session you will rank the segments of the N-122 in Soria by accident rate. Before writing
any merge, model the data.

| File | Content |
|---|---|
| `model/segments.csv` | Tramo catalogue of the N-122: id, PK range, junctions at each end. |
| `model/stations.csv` | Counting stations: type (permanent or secondary), road, PK, and the segment they count. |
| `model/accidents.csv` | Accidents with victims on the N-122, 2016–2019, from the DGT microdata. |

In [ ]:
segments13 = pd.read_csv(DATA + "model/segments.csv")
stations13 = pd.read_csv(DATA + "model/stations.csv")
accidents13 = pd.read_csv(DATA + "model/accidents.csv")

### Task 3.1 · Write the data model


Fill in the table, then sketch the relations between the tables with their cardinality
(1:1, 1:n, n:n). Say which columns link them.

| Table | Entity | Observation (one row is…) | Key | Links to | Cardinality | Through |
|---|---|---|---|---|---|---|
| `segments` | | | | | | |
| `stations` | | | | | | |
| `accidents` | | | | | | |

In [ ]:
# your code here

### Task 3.2 · Attach the station to each segment


Add to each segment the code of its counting station. Use `validate=` with the cardinality
your model says.

In [ ]:
# your code here